# Spark Sandbox — Getting Started

## Service endpoints

| Service | URL | Notes |
|---|---|---|
| Spark Master UI | http://localhost:8080 | Cluster overview |
| Spark App UI | http://localhost:4040 | Active while a job runs |
| Kafka UI | http://localhost:8090 | Topics, messages, consumer groups |
| JupyterLab | http://localhost:8888 | token: `spark123` |
| PostgreSQL | `localhost:5432` | db: sandbox, user: spark, pwd: spark123 |

## Sections
1. SparkSession setup
2. Batch pipeline → PostgreSQL
3. DuckDB analytics on Parquet
4. Kafka producer
5. Spark Structured Streaming from Kafka

---
## 1 · SparkSession

In [1]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

SPARK_MASTER  = os.getenv('SPARK_MASTER',            'spark://spark-master:7077')
KAFKA_SERVERS = os.getenv('KAFKA_BOOTSTRAP_SERVERS', 'kafka:29092')
PG_URL        = f"jdbc:postgresql://{os.getenv('POSTGRES_HOST','postgres')}:5432/{os.getenv('POSTGRES_DB','sandbox')}"
PG_PROPS      = {'user': os.getenv('POSTGRES_USER','spark'), 'password': os.getenv('POSTGRES_PASSWORD','spark123'), 'driver': 'org.postgresql.Driver'}

spark = (
    SparkSession.builder
    .appName('sandbox')
    .master(SPARK_MASTER)
    .getOrCreate()
)

spark.sparkContext.setLogLevel('WARN')
print(f'Spark {spark.version} — master: {spark.sparkContext.master}')

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/17 21:04:27 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 4.0.3 — master: spark://spark-master:7077


---
## 2 · Batch pipeline

Pattern: **generate → transform → write PostgreSQL + Parquet**

In [2]:
# Read existing tables from PostgreSQL
users = spark.read.jdbc(PG_URL, 'users', properties=PG_PROPS)
txns  = spark.read.jdbc(PG_URL, 'transactions', properties=PG_PROPS)

users.printSchema()
txns.show(5)

root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- country: string (nullable = true)
 |-- created_at: timestamp (nullable = true)

+---+-------+------+-----------+---------+--------------------+
| id|user_id|amount|   category|   status|          created_at|
+---+-------+------+-----------+---------+--------------------+
|  1|      1|150.00|electronics|completed|2026-06-17 20:18:...|
|  2|      1| 25.50|       food|completed|2026-06-17 20:18:...|
|  3|      2| 89.99|   clothing|completed|2026-06-17 20:18:...|
|  4|      3|200.00|electronics|completed|2026-06-17 20:18:...|
|  5|      2| 12.00|       food|completed|2026-06-17 20:18:...|
+---+-------+------+-----------+---------+--------------------+
only showing top 5 rows


In [3]:
from faker import Faker
import random

# Generate synthetic transactions
fake   = Faker()
CATS   = ['electronics', 'food', 'clothing', 'travel', 'health']
STATUS = ['completed', 'pending', 'refunded']

rows = [
    (random.randint(1, 5), round(random.uniform(5, 500), 2),
     random.choice(CATS), random.choice(STATUS),
     fake.date_time_between('-30d', 'now'))
    for _ in range(200)
]

schema = StructType([
    StructField('user_id',    IntegerType()),
    StructField('amount',     DoubleType()),
    StructField('category',   StringType()),
    StructField('status',     StringType()),
    StructField('created_at', TimestampType()),
])

df = spark.createDataFrame(rows, schema)

# Enrich with user country
enriched = (
    df.join(users.select('id', 'country'), df.user_id == users.id)
      .drop('id')
      .withColumn('date', F.to_date('created_at'))
)

enriched.show(5)

+-------+------+-----------+---------+--------------------+-------+----------+
|user_id|amount|   category|   status|          created_at|country|      date|
+-------+------+-----------+---------+--------------------+-------+----------+
|      1|141.35|electronics| refunded|2026-06-05 18:42:...|     US|2026-06-05|
|      1|214.23|       food| refunded|2026-05-27 17:22:...|     US|2026-05-27|
|      1| 98.15|electronics|completed|2026-05-21 15:07:...|     US|2026-05-21|
|      1|330.82|   clothing|completed|2026-05-30 23:15:...|     US|2026-05-30|
|      1|350.06|electronics|completed|2026-05-22 06:47:...|     US|2026-05-22|
+-------+------+-----------+---------+--------------------+-------+----------+
only showing top 5 rows


In [4]:
# Daily category summary
summary = (
    enriched
    .filter(F.col('status') == 'completed')
    .groupBy('date', 'category')
    .agg(
        F.round(F.sum('amount'), 2).alias('total_amount'),
        F.count('*').alias('tx_count'),
    )
    .orderBy('date', 'category')
)

summary.show(10)

+----------+-----------+------------+--------+
|      date|   category|total_amount|tx_count|
+----------+-----------+------------+--------+
|2026-05-19|   clothing|      798.37|       3|
|2026-05-19|electronics|      201.55|       1|
|2026-05-20|electronics|       354.6|       1|
|2026-05-20|       food|      223.88|       1|
|2026-05-21|   clothing|      637.17|       3|
|2026-05-21|electronics|      465.41|       2|
|2026-05-22|electronics|      750.28|       2|
|2026-05-22|     travel|      249.84|       1|
|2026-05-23|electronics|      503.21|       2|
|2026-05-23|     travel|      343.99|       1|
+----------+-----------+------------+--------+
only showing top 10 rows


In [5]:
# Write to PostgreSQL
summary.withColumnRenamed('date', 'summary_date') \
       .write.jdbc(PG_URL, 'daily_summary', mode='overwrite', properties=PG_PROPS)

# Write to Parquet (for DuckDB section)
enriched.write.mode('overwrite').parquet('/home/jupyter/data/output/transactions')

print('Done writing.')

Done writing.


---
## 3 · DuckDB analytics

DuckDB can query Parquet files directly — great for fast ad-hoc analytics after a Spark job.

In [6]:
import duckdb

con = duckdb.connect()  # in-memory

# Register the Parquet dataset written by Spark
con.execute("CREATE VIEW transactions AS SELECT * FROM read_parquet('/home/jupyter/data/output/transactions/**/*.parquet')")

con.execute('SELECT * FROM transactions LIMIT 5').df()

,user_id,amount,category,status,created_at,country,date
0,1,141.35,electronics,refunded,2026-06-05 18:42:24.938357,US,2026-06-05
1,1,214.23,food,refunded,2026-05-27 17:22:58.548758,US,2026-05-27
2,1,98.15,electronics,completed,2026-05-21 15:07:14.363826,US,2026-05-21
3,1,330.82,clothing,completed,2026-05-30 23:15:31.567506,US,2026-05-30
4,1,350.06,electronics,completed,2026-05-22 06:47:47.160694,US,2026-05-22


In [7]:
# Revenue by country & category
result = con.execute("""
    SELECT
        country,
        category,
        ROUND(SUM(amount), 2)  AS total_revenue,
        COUNT(*)               AS tx_count,
        ROUND(AVG(amount), 2)  AS avg_amount
    FROM transactions
    WHERE status = 'completed'
    GROUP BY ALL
    ORDER BY total_revenue DESC
""").df()

result

,country,category,total_revenue,tx_count,avg_amount
0,MX,clothing,2557.26,9,284.14
1,MX,health,1690.57,5,338.11
2,SG,clothing,1596.85,7,228.12
3,US,clothing,1356.12,4,339.03
4,SG,electronics,1337.55,5,267.51
5,DE,electronics,1279.29,4,319.82
6,US,electronics,1135.70,7,162.24
7,MX,travel,893.59,3,297.86
8,UK,food,793.68,3,264.56
9,DE,travel,764.39,2,382.20


In [ ]:
import matplotlib.pyplot as plt

pivot = result.pivot(index='category', columns='country', values='total_revenue').fillna(0)
pivot.plot(kind='bar', figsize=(10, 5), title='Revenue by Category & Country')
plt.tight_layout()
plt.show()

---
## 4 · Kafka producer

Send simulated events that the streaming job will consume.

In [8]:
import json, time, random, threading
from kafka import KafkaProducer
from faker import Faker
from datetime import datetime, timezone

fake = Faker()
CATS = ['electronics', 'food', 'clothing', 'travel', 'health']

producer = KafkaProducer(
    bootstrap_servers=KAFKA_SERVERS,
    value_serializer=lambda v: json.dumps(v).encode(),
)

def send_events(n=50, delay=0.2):
    for i in range(n):
        event = {
            'event_id':  fake.uuid4(),
            'user_id':   random.randint(1, 5),
            'amount':    round(random.uniform(5, 500), 2),
            'category':  random.choice(CATS),
            'ts':        datetime.now(timezone.utc).isoformat(),
        }
        producer.send('transactions', value=event)
        time.sleep(delay)
    producer.flush()
    print(f'Sent {n} events.')

# Send events in a background thread so the notebook stays responsive
t = threading.Thread(target=send_events, args=(100, 0.1), daemon=True)
t.start()
print('Producer started — check Kafka UI at http://localhost:8090')

Producer started — check Kafka UI at http://localhost:8090


---
## 5 · Spark Structured Streaming

Consume `transactions` topic, aggregate over a 1-minute tumbling window, write to console and PostgreSQL.

In [9]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType, TimestampType

event_schema = StructType([
    StructField('event_id', StringType()),
    StructField('user_id',  IntegerType()),
    StructField('amount',   DoubleType()),
    StructField('category', StringType()),
    StructField('ts',       StringType()),
])

raw = (
    spark.readStream
    .format('kafka')
    .option('kafka.bootstrap.servers', KAFKA_SERVERS)
    .option('subscribe', 'transactions')
    .option('startingOffsets', 'earliest')
    .load()
)

events = (
    raw
    .select(F.from_json(F.col('value').cast('string'), event_schema).alias('e'))
    .select('e.*')
    .withColumn('event_time', F.to_timestamp('ts'))
)

events.printSchema()

root
 |-- event_id: string (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- amount: double (nullable = true)
 |-- category: string (nullable = true)
 |-- ts: string (nullable = true)
 |-- event_time: timestamp (nullable = true)



In [10]:
# 1-minute tumbling window aggregation with 10-second watermark
windowed = (
    events
    .withWatermark('event_time', '10 seconds')
    .groupBy(
        F.window('event_time', '1 minute'),
        'category',
    )
    .agg(
        F.round(F.sum('amount'), 2).alias('total_amount'),
        F.count('*').alias('tx_count'),
    )
    .select(
        F.col('window.start').alias('window_start'),
        F.col('window.end').alias('window_end'),
        'category',
        'total_amount',
        'tx_count',
    )
)

In [11]:
# Print to console — swap outputMode/sink for production use
query = (
    windowed.writeStream
    .outputMode('update')
    .format('console')
    .option('truncate', False)
    .option('checkpointLocation', '/home/jupyter/data/checkpoints/streaming_demo')
    .trigger(processingTime='15 seconds')
    .start()
)

print(f'Streaming query started: {query.id}')
print('Results appear in the Spark UI → Structured Streaming tab.')
print('Run query.stop() to halt.')

Streaming query started: e2705ea6-7802-48f7-a060-80cf82cf1271
Results appear in the Spark UI → Structured Streaming tab.
Run query.stop() to halt.


26/06/17 21:05:44 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


Sent 100 events.
-------------------------------------------
Batch: 0
-------------------------------------------
+-------------------+-------------------+-----------+------------+--------+
|window_start       |window_end         |category   |total_amount|tx_count|
+-------------------+-------------------+-----------+------------+--------+
|2026-06-17 21:05:00|2026-06-17 21:06:00|health     |3066.06     |17      |
|2026-06-17 21:05:00|2026-06-17 21:06:00|food       |4840.75     |18      |
|2026-06-17 21:05:00|2026-06-17 21:06:00|clothing   |6282.01     |25      |
|2026-06-17 21:05:00|2026-06-17 21:06:00|travel     |4827.61     |17      |
|2026-06-17 21:05:00|2026-06-17 21:06:00|electronics|5409.13     |20      |
+-------------------+-------------------+-----------+------------+--------+

-------------------------------------------
Batch: 1
-------------------------------------------
+-------------------+-------------------+-----------+------------+--------+
|window_start       |window_

In [12]:
# Stop when done
query.stop()
print('Stream stopped.')

Stream stopped.


26/06/17 21:06:01 ERROR WriteToDataSourceV2Exec: Data source write support MicroBatchWrite[epoch: 2, writer: ConsoleWriter[numRows=20, truncate=false]] is aborting.
26/06/17 21:06:01 ERROR WriteToDataSourceV2Exec: Data source write support MicroBatchWrite[epoch: 2, writer: ConsoleWriter[numRows=20, truncate=false]] aborted.
26/06/17 21:06:01 WARN DAGScheduler: Failed to cancel job group c86af264-f91b-4e82-82df-433536f5a4fd. Cannot find active jobs for it.


26/06/17 21:06:01 WARN TaskSetManager: Lost task 151.0 in stage 43.0 (TID 574) (172.20.0.7 executor 0): TaskKilled (Stage cancelled: [SPARK_JOB_CANCELLED] Job 19 cancelled Query [id = e2705ea6-7802-48f7-a060-80cf82cf1271, runId = c86af264-f91b-4e82-82df-433536f5a4fd] was stopped SQLSTATE: XXKDA)
26/06/17 21:06:01 WARN TaskSetManager: Lost task 150.0 in stage 43.0 (TID 573) (172.20.0.7 executor 0): TaskKilled (Stage cancelled: [SPARK_JOB_CANCELLED] Job 19 cancelled Query [id = e2705ea6-7802-48f7-a060-80cf82cf1271, runId = c86af264-f91b-4e82-82df-433536f5a4fd] was stopped SQLSTATE: XXKDA)
26/06/17 21:30:41 ERROR StandaloneSchedulerBackend: Application has been killed. Reason: Master removed our application: KILLED
26/06/17 21:30:41 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exiting due to error from cluster scheduler: Master removed our application: KILLED
	at org.apache.spark.errors.SparkCoreErrors$.clusterSchedulerError(SparkCoreErrors.scala:295)
	at org.apache.spark.